# nb12: Residual Analysis (a.k.a. "Honest Outlier Removal")

**Read this carefully before running.**

## What this notebook does

Identifies compounds that the C6 XGBoost model predicts poorly (high LOO residual), removes them, and re-evaluates. Shows you R² with each compound removed, then with top-k removed jointly.

## Why this is dangerous methodology

Removing the worst-predicted compounds and reporting the improved R² is **circular** — you're using the model to choose which data points to keep, then evaluating the model on the kept points. R² will go up by construction. A reviewer will see through this immediately.

## How we make it defensible

1. **We always report both numbers.** Original 99-compound LOO R² is preserved alongside the trimmed result. You never "lose" the baseline.

2. **We test stability across feature sets.** If a compound is high-residual for C6 only, removing it is overfitting. If it's high-residual across multiple feature sets, it's a genuine chemistry outlier.

3. **We test stability across seeds.** XGBoost is stochastic. If a compound is high-residual for seed=42 only, that's noise. If it's high-residual for 5+ seeds, it's a stable signal.

4. **We identify what the removed compounds ARE chemically.** Their formulas, alpha_R values, lattice constants, k-paths. If they cluster (e.g., all heavy halides, all Janus structures), the removal is a *finding* about the model's domain of applicability — that's publishable. If they're random, the gain is noise.

## What you can honestly report

> "We computed LOO residuals on the C6 XGBoost model and identified N compounds with errors > 2.5×MAE. These compounds correspond to <chemistry pattern>, suggesting our descriptor set does not capture <missing physics>. After excluding these as out-of-domain, the model achieves R² = X.XXX on the remaining 99-N compounds."

That framing is defensible. It tells the reader: here's where our model works, here's where it doesn't, here's the honest number for each region.

## What you should NOT report

> "After outlier removal, our model achieves R² = X.XXX." (without specifying which compounds were removed and why)

That gets you laughed out of a defense.


## Cell 1: Imports & paths

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from time import time

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join('..'))
OLD_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors_old.csv')
NEW_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors.csv')
RESULTS_DIR = os.path.join('.', 'nb12_residual-results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup OK. Results dir:', RESULTS_DIR)


## Cell 2: Load merged data, collapse to 99 compounds, baseline

In [ ]:
df_old = pd.read_csv(OLD_CSV)
df_new = pd.read_csv(NEW_CSV)

ID_COLS = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
TARGET = 'Rashba_parameter'

df_merged = df_old[ID_COLS].copy()
old_features = [c for c in df_old.columns if c not in ID_COLS]
new_features = [c for c in df_new.columns if c not in ID_COLS]
overlap = set(old_features) & set(new_features)
for col in old_features:
    df_merged[f'old_{col}' if col in overlap else col] = df_old[col].values
for col in new_features:
    df_merged[f'new_{col}' if col in overlap else col] = df_new[col].values

idx_max = df_merged.groupby('uid')[TARGET].idxmax()
df_99 = df_merged.loc[idx_max].reset_index(drop=True)
y_99 = df_99[TARGET].values
print(f'99-row df: {df_99.shape}')

XGB_REG_PARAMS = dict(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=1.0,
    random_state=42, verbosity=0,
)

BASELINE_RAW = ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean',
                'pmid_afs_gauss_std', 'kpath_angle_deg', 'ehull']

def resolve(name, cols):
    if name in cols: return name
    if f'old_{name}' in cols: return f'old_{name}'
    if f'new_{name}' in cols: return f'new_{name}'
    raise KeyError(name)

BASELINE = [resolve(n, df_99.columns) for n in BASELINE_RAW]

def loo_preds(features, df, y, seed=42):
    """Return LOO predictions and R2 / MAE."""
    X = df[features].fillna(0).values
    params = dict(XGB_REG_PARAMS); params['random_state'] = seed
    model = XGBRegressor(**params)
    y_pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
    return y_pred, r2_score(y, y_pred), mean_absolute_error(y, y_pred)


y_pred_base, r2_base, mae_base = loo_preds(BASELINE, df_99, y_99)
residuals_base = y_99 - y_pred_base
abs_resid = np.abs(residuals_base)

print(f'Baseline LOO R2 = {r2_base:.4f}, MAE = {mae_base:.4f}')
print(f'Residual stats: mean abs = {abs_resid.mean():.3f}, max = {abs_resid.max():.3f}, std = {residuals_base.std():.3f}')


## Cell 3: Identify high-residual compounds (single-seed, C6 features)

Rank compounds by absolute residual. The top-k will be candidates for removal.

**Caveat:** this ranking is based on a single seed and a single feature set. We'll test stability across both in subsequent cells.

In [ ]:
# Build a residual table
resid_df = df_99[['uid', 'Formula', 'kpath', TARGET]].copy()
resid_df['y_pred'] = y_pred_base
resid_df['residual'] = residuals_base
resid_df['abs_residual'] = abs_resid
resid_df = resid_df.sort_values('abs_residual', ascending=False).reset_index(drop=True)

print('Top 15 high-residual compounds (single seed, C6):')
print(resid_df.head(15).to_string(index=False))

resid_df.to_csv(os.path.join(RESULTS_DIR, 'residuals_C6_seed42.csv'), index=False)


## Cell 4: Stability check across seeds

Run LOO with seeds [0..9]. For each seed, get the residual rank of each compound. A compound is a *stable* outlier if it ranks in the top 10 across most seeds.

In [ ]:
print('Running 10 seeds for residual stability...')
all_residuals = pd.DataFrame({'uid': df_99['uid'], 'formula': df_99['Formula']})

for seed in range(10):
    y_pred, _, _ = loo_preds(BASELINE, df_99, y_99, seed=seed)
    all_residuals[f'abs_resid_seed{seed}'] = np.abs(y_99 - y_pred)

# For each seed, compute rank (1 = highest residual)
rank_cols = []
for seed in range(10):
    rank_col = f'rank_seed{seed}'
    all_residuals[rank_col] = all_residuals[f'abs_resid_seed{seed}'].rank(ascending=False).astype(int)
    rank_cols.append(rank_col)

all_residuals['mean_rank'] = all_residuals[rank_cols].mean(axis=1)
all_residuals['mean_abs_resid'] = all_residuals[[f'abs_resid_seed{s}' for s in range(10)]].mean(axis=1)
all_residuals['n_times_top10'] = (all_residuals[rank_cols] <= 10).sum(axis=1)

# Sort by stability of being in top 10
stable_outliers = all_residuals.sort_values('n_times_top10', ascending=False).reset_index(drop=True)

print('\nTop 15 STABLE outliers (top 10 across most seeds):')
print(stable_outliers[['uid', 'formula', 'mean_abs_resid', 'mean_rank', 'n_times_top10']].head(15).to_string(index=False))

stable_outliers.to_csv(os.path.join(RESULTS_DIR, 'residuals_C6_multiseed.csv'), index=False)


## Cell 5: Stability check across feature sets

We've been measuring residuals only on C6. Now check: are these compounds also high-residual when we use *different* feature sets? If yes, they're genuine chemistry outliers. If they're bad only for C6, removing them is overfitting C6.

We test 3 alternative feature sets that have been previously identified as decent in nb7.

In [ ]:
# Alternative feature sets from nb7 (good but not C6)
ALT_SETS = {
    '4feat_classification_winner': ['E_pfrac_VBM', 'pmid_afs_gauss_std', 'kpath_angle_deg', 'E_sfrac_CBM'],
    '5feat_minus_ehull': ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean', 'pmid_afs_gauss_std', 'kpath_angle_deg'],
    '7feat_C6_plus_z4': ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean', 'pmid_afs_gauss_std',
                        'kpath_angle_deg', 'ehull', 'max_Z4'],
}

resid_by_set = pd.DataFrame({'uid': df_99['uid'], 'formula': df_99['Formula']})
resid_by_set['abs_resid_C6'] = abs_resid

for name, raw_feats in ALT_SETS.items():
    try:
        feats = [resolve(f, df_99.columns) for f in raw_feats]
        y_pred, r2_alt, _ = loo_preds(feats, df_99, y_99)
        resid_by_set[f'abs_resid_{name}'] = np.abs(y_99 - y_pred)
        print(f'{name}: LOO R2 = {r2_alt:.4f}')
    except KeyError as e:
        print(f'{name}: SKIPPED (missing column {e})')

# Mean abs residual across feature sets
resid_cols = [c for c in resid_by_set.columns if c.startswith('abs_resid_')]
resid_by_set['mean_abs_resid_across_sets'] = resid_by_set[resid_cols].mean(axis=1)

# How many feature sets does this compound rank in top 10?
n_top10 = pd.Series(0, index=resid_by_set.index)
for col in resid_cols:
    rank = resid_by_set[col].rank(ascending=False)
    n_top10 += (rank <= 10).astype(int)
resid_by_set['n_sets_top10'] = n_top10

cross_set_outliers = resid_by_set.sort_values('n_sets_top10', ascending=False).reset_index(drop=True)
print('\nTop 15 outliers across feature sets:')
display_cols = ['uid', 'formula', 'mean_abs_resid_across_sets', 'n_sets_top10'] + resid_cols
print(cross_set_outliers[display_cols].head(15).to_string(index=False))

cross_set_outliers.to_csv(os.path.join(RESULTS_DIR, 'residuals_cross_set.csv'), index=False)


## Cell 6: The actual removal experiment

For k = 1, 2, 3, 5, 7, 10:
- Define the "outlier set" as the top-k stable outliers (using mean_rank from Cell 4)
- Drop those compounds, re-run LOO on the remaining (99 - k) compounds
- Report R² with and without

This shows you *exactly* what the gain looks like, with no hiding the original number.

In [ ]:
# Use stable_outliers (from Cell 4) ranked by n_times_top10
candidates_to_remove_in_order = stable_outliers['uid'].tolist()

removal_results = []
removal_results.append({
    'k_removed': 0,
    'n_remaining': 99,
    'r2': r2_base,
    'mae': mae_base,
    'removed_uids': '',
    'removed_formulas': '',
})

for k in [1, 2, 3, 5, 7, 10]:
    removed_uids = candidates_to_remove_in_order[:k]
    keep_mask = ~df_99['uid'].isin(removed_uids)
    df_keep = df_99[keep_mask].reset_index(drop=True)
    y_keep = df_keep[TARGET].values

    y_pred_k, r2_k, mae_k = loo_preds(BASELINE, df_keep, y_keep)

    removed_formulas = df_99[df_99['uid'].isin(removed_uids)]['Formula'].tolist()
    removal_results.append({
        'k_removed': k,
        'n_remaining': len(df_keep),
        'r2': r2_k,
        'mae': mae_k,
        'removed_uids': ', '.join(removed_uids),
        'removed_formulas': ', '.join(removed_formulas),
    })

removal_df = pd.DataFrame(removal_results)
print('Removal experiment (using stable cross-seed top outliers):')
print(removal_df[['k_removed', 'n_remaining', 'r2', 'mae']].to_string(index=False))
print()
print('Removed compounds at each step:')
for _, row in removal_df.iterrows():
    if row['k_removed'] > 0:
        print(f'  k={row["k_removed"]:2d}: {row["removed_formulas"]}')

removal_df.to_csv(os.path.join(RESULTS_DIR, 'removal_experiment.csv'), index=False)


## Cell 7: Sanity check — does removing RANDOM compounds give similar gains?

This is the most important cell. If removing 5 random compounds gives a similar R² gain to removing the 5 worst-residual ones, **the gain is statistical artifact**, not real outlier removal.

For each k in [1, 3, 5, 10], we:
- Remove k *random* compounds, re-run LOO, record R²
- Repeat 50 times
- Compare the distribution of "random removal" R² to "outlier removal" R²

If outlier removal R² is at the high end of the random distribution → the gain is mostly statistical.
If outlier removal R² is *above* the random distribution's max → genuine outlier signal.

In [ ]:
rng = np.random.default_rng(42)
n_random_trials = 50

random_results = []
for k in [1, 3, 5, 10]:
    r2s = []
    for trial in range(n_random_trials):
        random_drop_idx = rng.choice(99, size=k, replace=False)
        keep_mask = ~df_99.index.isin(random_drop_idx)
        df_keep = df_99[keep_mask].reset_index(drop=True)
        y_keep = df_keep[TARGET].values
        _, r2, _ = loo_preds(BASELINE, df_keep, y_keep)
        r2s.append(r2)

    outlier_r2 = removal_df[removal_df['k_removed'] == k]['r2'].iloc[0]
    pct_above = 100 * sum(r2_rand >= outlier_r2 for r2_rand in r2s) / n_random_trials

    random_results.append({
        'k_removed': k,
        'random_r2_mean': np.mean(r2s),
        'random_r2_std': np.std(r2s),
        'random_r2_min': np.min(r2s),
        'random_r2_max': np.max(r2s),
        'outlier_r2': outlier_r2,
        'pct_random_above_outlier': pct_above,
    })

rand_df = pd.DataFrame(random_results)
print('Random removal vs outlier removal:')
print(rand_df.to_string(index=False))
print()
print('Interpretation:')
print('  - If outlier_r2 > random_r2_max: outlier removal is genuinely better than random')
print('  - If outlier_r2 ~ random_r2_mean: outlier removal is noise (your gain came from just removing data)')
print('  - pct_random_above_outlier: fraction of random trials that beat outlier removal')
print('    > 30%: gain is mostly statistical, not real outlier signal')
print('    < 5%: gain is real')

rand_df.to_csv(os.path.join(RESULTS_DIR, 'random_vs_outlier_removal.csv'), index=False)


## Cell 8: Chemistry inspection of removed compounds

What ARE the compounds that the model fails on? Look for patterns. If they all share a chemistry feature (heavy halides, magnetic elements, layered chalcogenides, Janus structures, large alpha_R values, edge-of-distribution stability), that's a genuine domain-of-applicability finding.

In [ ]:
top_outliers = stable_outliers.head(10).copy()
# Bring back useful info from df_99
extra_info = df_99[['uid', 'Formula', 'kpath', TARGET, 'ehull']].rename(columns={TARGET: 'alpha_R'})
top_outliers = top_outliers[['uid', 'mean_abs_resid', 'n_times_top10']].merge(extra_info, on='uid', how='left')

print('Top 10 stable outliers — chemistry inspection:')
print(top_outliers.to_string(index=False))
print()

# Patterns to look for
print('Pattern analysis:')
formulas = top_outliers['Formula'].tolist()
print(f'  Formulas: {formulas}')

# Element frequency
from collections import Counter
all_elements = []
for f in formulas:
    if isinstance(f, str) and f.strip():
        # Crude split - works for most cases
        import re
        elems = re.findall(r'[A-Z][a-z]?', f)
        all_elements.extend(elems)
elem_counts = Counter(all_elements)
print(f'  Element frequency in outliers: {dict(elem_counts.most_common(10))}')

# Compare to baseline distribution
all_elements_99 = []
for f in df_99['Formula'].dropna():
    elems = re.findall(r'[A-Z][a-z]?', f)
    all_elements_99.extend(elems)
elem_counts_99 = Counter(all_elements_99)
print(f'\n  For comparison, top elements across all 99: {dict(elem_counts_99.most_common(10))}')

# alpha_R distribution
print(f'\n  alpha_R of outliers: {top_outliers["alpha_R"].tolist()}')
print(f'  alpha_R range across all 99: [{y_99.min():.3f}, {y_99.max():.3f}]')
print(f'  alpha_R mean across all 99: {y_99.mean():.3f}')
print(f'  alpha_R mean of outliers:    {top_outliers["alpha_R"].mean():.3f}')

print()
print('THINK ABOUT THIS:')
print('- Are the outliers concentrated at the high or low end of alpha_R?')
print('  If yes -> the model fails at extreme values. This is real, document it.')
print('- Do they share a specific element (e.g., all contain Bi, or W)?')
print('  If yes -> something about that element is missed by the descriptors.')
print('- Are they all from a specific space group / layer type?')
print('  If yes -> structural feature missing.')
print('- If patterns are random -> gain from removal is noise. Do not report.')


## Cell 9: Final verdict & honest summary

Pull together all the diagnostics into one summary statement you can paste into your DDP report or use to talk with your prof.

In [ ]:
print('=' * 70)
print('  HONEST SUMMARY — what to actually report')
print('=' * 70)
print()
print(f'Baseline (C6, all 99 compounds): R2 = {r2_base:.4f}, MAE = {mae_base:.4f}')
print()

# Pull max gain from removal experiment
removal_with_gain = removal_df.copy()
removal_with_gain['delta_r2'] = removal_with_gain['r2'] - r2_base
print('R2 vs k removed:')
for _, row in removal_with_gain.iterrows():
    if row['k_removed'] == 0: continue
    print(f'  k={int(row["k_removed"]):2d} removed -> R2 = {row["r2"]:.4f} ({row["delta_r2"]:+.4f})')

print()
print('Random-removal control (Cell 7):')
print(rand_df[['k_removed', 'outlier_r2', 'random_r2_mean', 'random_r2_max', 'pct_random_above_outlier']].to_string(index=False))

print()
print('---')
print('VERDICT GUIDE:')
print('  Look at "pct_random_above_outlier" for k=5 or k=10.')
print('  If < 10%: outlier removal is a real signal. Report with caveats.')
print('  If 10-30%: marginal. Investigate chemistry patterns (Cell 8).')
print('  If > 30%: noise. Do not report the trimmed number; report only baseline.')
print()
print('IF YOU REPORT TRIMMED RESULTS, the framing must include:')
print('  1. The original 99-compound number')
print('  2. Which compounds were removed (formulas, uids)')
print('  3. Why they were considered outliers (chemistry, alpha_R extremes, etc.)')
print('  4. The random-removal control numbers')
print('Do NOT report a trimmed number without these four pieces.')


## Notes for next steps

1. **The honest version of this notebook is the one your prof wants.** If you bring her this with the random-removal control and chemistry inspection, she will respect it. If you bring her "we removed 10 compounds and got 0.78", she will rightfully push back.

2. **"Outliers in target space" is what your prof originally asked for.** Your alpha_R > 3.5 cutoff was that. If it didn't help, the prediction errors aren't concentrated at high alpha_R — meaning the model isn't biased toward extreme values, it's biased *somewhere else*. Cell 8 will tell you where.

3. **If random removal gives most of the gain**, the right action is not to give up — it's to recognize that you're underfitting (with 99 compounds, removing any 5 reduces variance from those 5 noisy points). The fix isn't outlier removal; it's better features.

4. **A truly defensible outlier removal** uses a criterion *defined before training*: e.g., "we exclude compounds with ehull > X eV/atom because they are thermodynamically unstable and our DFT data is unreliable for them". That's a *physical* criterion, not a model-output criterion. If Cell 8 reveals the outliers cluster around high ehull, that's grounds for a defensible criterion.

5. **One more thing to try if results are still bad:** the issue may not be outliers, it may be that some compounds have very small alpha_R values where measurement / DFT noise dominates. A weighted-loss XGBoost (weighting samples by alpha_R) is a defensible alternative — it doesn't drop data, it just down-weights the noisy small-alpha part.
